# Long-Term Memory with LangMem — Hot Path Processing

## What this notebook teaches

This notebook demonstrates the **hot path** approach to memory management — the synchronous alternative to background processing.

## Background vs Hot Path

| | Background | Hot Path |
|---|---|---|
| Memory extraction timing | After response is returned | Before response is generated |
| Response latency | Fast (user doesn't wait) | Slower (user waits for extraction) |
| Memories usable in current reply | No | Yes |
| Best for | High-volume chat, long sessions | Personalisation-critical apps |

## How it works

```
User sends message
       │
       ▼
 Search store ──► Retrieve relevant memories from past sessions
       │
       ▼
 LLM (with memories as context) ──► Generates personalised response
       │
       ▼
 Memory Manager (LLM) ──► Extracts new facts from this turn
       │
       ▼
 InMemoryStore ──► Saves updated memories
       │
       ▼
 Response returned to user
```

**Key idea:** Memory extraction runs *before* the response is returned, so the LLM can use freshly extracted memories to personalise its reply within the same turn.

In [1]:
from langchain.chat_models import init_chat_model
from langgraph.func import entrypoint
from langgraph.store.memory import InMemoryStore
from langmem import create_memory_store_manager

## Step 1: Create the Memory Store

`InMemoryStore` is a vector database that lives in RAM. It stores memories as embeddings so you can do **semantic search** — find memories by meaning, not just exact keywords.

- `dims=768` — the vector size produced by Gemini's embedding model
- `embed` — the model used to convert text memories into vectors

In [2]:
store = InMemoryStore(
    index={
        "dims": 768,                                   # Must match the embedding model's output size
        "embed": "google_genai:gemini-embedding-001",  # Model used to embed memories into vectors
    }
)

## Step 2: Set Up the Memory Manager

**`create_memory_store_manager`** is an LLM-powered "librarian". It reads a conversation and decides which facts are worth remembering.

Unlike the background approach, there is **no `ReflectionExecutor`** here. The memory manager is called directly with `await memory_manager.ainvoke(...)` inside the chat function, so extraction is synchronous and the store is updated before the response is returned.

In [3]:
# LLM-powered memory extractor: reads conversations and picks out memorable facts.
# Called directly (not via ReflectionExecutor) so extraction is synchronous.
memory_manager = create_memory_store_manager(
    "google_genai:gemini-2.5-flash",
    namespace=("memories",),  # Namespace acts like a folder path for organising memories
)

## Step 3: Build the Chat Function

The hot path chat function has **three stages** inside a single turn:

1. **Retrieve** — search the store for memories relevant to the current message
2. **Respond** — generate a reply using those memories as context
3. **Extract** — run the memory manager on this turn so new facts are saved

Because stage 3 completes before the function returns, any facts learned in this turn are immediately available to the *next* call — even though the user only sees the response at the end.

In [4]:
llm = init_chat_model("google_genai:gemini-2.5-flash")

@entrypoint(store=store)  # Wires the store into LangGraph's runtime context
async def chat(message: str):
    # --- Stage 1: Retrieve relevant memories from past sessions ---
    # store.asearch() does a semantic (vector) search against the message text.
    # This finds memories whose *meaning* is related, not just keyword matches.
    memories = await store.asearch(("memories",), query=message)

    # Format retrieved memories as a readable block to inject into the prompt.
    # If there are no memories yet, this section is omitted entirely.
    memory_context = ""
    if memories:
        facts = "\n".join(
            f"- {m.value['content']['content']}" for m in memories
        )
        memory_context = f"\n\nWhat you remember about this user:\n{facts}"

    # --- Stage 2: Generate a personalised response ---
    # Memories are injected into the system prompt so the LLM can reference them.
    system_prompt = (
        "You are a helpful assistant with a long-term memory."
        " Use the memories provided to personalise your response."
        + memory_context
    )
    response = await llm.ainvoke([
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": message},
    ])

    # --- Stage 3: Extract and store new memories from this turn (hot path) ---
    # Unlike the background approach, we AWAIT this call — it blocks until
    # the memory manager has finished writing to the store.
    # Trade-off: the user waits a bit longer, but memories are immediately
    # available on the very next call.
    await memory_manager.ainvoke(
        {
            "messages": [
                {"role": "user",      "content": message},
                response,             # AIMessage returned by llm.ainvoke()
            ]
        },
        config={"store": store},      # Explicit store config required for hot-path invocations
    )

    return response.content

## Step 4: First Turn — Introduce Memorable Facts

Send a message that contains facts the memory manager should pick up.
Because we are on the hot path, extraction completes **before** this cell finishes executing — no `asyncio.sleep()` required.

In [5]:
response = await chat.ainvoke("I like dogs. My dog's name is Fido.")
print("Bot:", response)

Bot: That's wonderful! Dogs are amazing companions, and it's lovely to hear you have Fido in your life. What kind of dog is Fido?


## Step 5: Inspect Memories Immediately

Unlike background processing, we do **not** need to sleep here.
The `await memory_manager.ainvoke(...)` in the chat function already completed,
so the store is fully updated.

In [6]:
# No asyncio.sleep() needed — memories are already written.
memories = store.search(("memories",))

if memories:
    print(f"✓ Found {len(memories)} stored memory/memories:\n")
    for i, mem in enumerate(memories, 1):
        print(f"  [{i}] {mem.value}")
else:
    print("No memories found.")

✓ Found 1 stored memory/memories:

  [1] {'kind': 'Memory', 'content': {'content': 'The user likes dogs.'}}


## Step 6: Second Turn — Personalisation in Action

Send a second message. Watch how the bot references Fido even though you didn't mention him — it retrieved that memory in Stage 1 of this turn.

In [7]:
response = await chat.ainvoke("What should I name my new pet?")
print("Bot:", response)

NumPy not found in the current Python environment. The InMemoryStore will use a pure Python implementation for vector operations, which may significantly impact performance, especially for large datasets or frequent searches. For optimal speed and efficiency, consider installing NumPy: pip install numpy


Bot: That's so exciting! Naming a new pet is one of the best parts.

To give you the best suggestions, could you tell me a little more about your new companion? For example:

*   **What kind of pet is it?** (e.g., dog, cat, hamster, bird, etc.)
*   **Is it male or female?**
*   **Does it have any distinctive features or a particular personality yet?** (e.g., fluffy, playful, calm, energetic)

In the meantime, here are some general ideas across different styles to get you started:

**Classic & Sweet:**
*   Bella
*   Charlie
*   Daisy
*   Max
*   Lucy
*   Cooper
*   Milo
*   Luna

**Playful & Unique:**
*   Ziggy
*   Pixel
*   Waffles
*   Gizmo
*   Pippin
*   Juno
*   Scout
*   Mocha

**Nature & Strong:**
*   Willow
*   River
*   Forest
*   Rocky
*   Hazel
*   Atlas
*   Storm
*   Raven

Let me know more details, and I can give you some more tailored suggestions!


## Step 7: Add More Facts and Verify Accumulation

Each turn extracts new memories and merges them with existing ones. The memory manager deduplicates and updates facts rather than blindly appending.

In [8]:
response = await chat.ainvoke("I also like cats. My cat's name is Whiskers.")
print("Bot:", response)

# Inspect the updated store — should now contain facts about both pets
print("\n--- Memories after second fact turn ---")
memories = store.search(("memories",))
for i, mem in enumerate(memories, 1):
    print(f"  [{i}] {mem.value}")

Bot: That's great to know! So you like cats too, and your cat's name is Whiskers. What a lovely name! I'll be sure to remember that you like both dogs and cats, and that Whiskers is your feline friend.

--- Memories after second fact turn ---
  [1] {'kind': 'Memory', 'content': {'content': "The user likes dogs and cats. Their cat's name is Whiskers."}}
